# 챕터 2 — 무역의 계절성: 한국 수출입의 월별 리듬 (2007–2025)

1년 안에서 수출입이 **어느 달에 몰리고 어느 달에 비는지**를 인과 해석 없이 서술하는 기술통계 노트북이다. 챕터 1이 연간 흐름을 봤다면, 여기서는 이 DB의 월별(231개월) 데이터로 계절 리듬을 그린다. 함께 있는 문서 `chapter02.md`가 이 결과를 이야기로 엮은 것이다.

## 이 데이터베이스에 대하여 (출처·집계 기준)

이 데이터베이스는 관세청이 OpenAPI(품목별 국가별 수출입실적(GW), https://www.data.go.kr/data/15100475/openapi.do)로 공개하는 월별 수출입 통계를 **2007년 1월부터 2026년 3월까지** 한데 모아 하나의 파일로 만든 것이다.

**수치의 집계 기준(관세청 정의).** 수출입 신고 통관 자료를 국가 및 HS Code(2·4·6·10단위)별로 집계한 국가별 품목별 무역통계다. 금액은 미화(USD)이며, 수출은 FOB(신고금액), 수입은 CIF(과세가격) 기준이다. 중량은 순중량(kg). 국가는 수출은 최종목적국, 수입은 원산국을 원칙으로 하며 무역통계부호상 ISO 코드로 분류한다. 단순 통과물품이나 일시 반입·반출 물품은 제외된다(물적 자원의 증감이 없으므로). 통계는 매월 수출입 신고의 정정·취하를 반영해 전월까지 자료를 현행화한다(주기 1개월).

## 0. 규칙과 함정

- **완전연도만**: 2007–2025(19년). 2026은 부분년이라 제외.
- **추세 제거**: 절대액이 아니라 "그 해 안에서의 비중(%)"으로 계절을 본다. 12달 합=100%, 균등=8.33%.
- **2월 주의**: 날수 28일 + 설날 근무일 감소가 섞인다(4절).
- 금액 단위는 미화 달러(USD).

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

MON = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

DB_PATH = os.path.join("data", "processed", "kcsdb.duckdb")
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"DB 없음: {DB_PATH} — Releases에서 받아 data/processed/ 에 배치")
con = duckdb.connect(DB_PATH, read_only=True)
def q(sql): return con.sql(sql).df()
print("연결 완료 —", f"{con.sql('SELECT COUNT(*) FROM fact_trade').fetchone()[0]:,}", "거래행")

## 3. 전체 계절 프로파일 — 월별 연간대비 비중(%)

In [ ]:
prof = q('''
    WITH ym AS (SELECT yyyymm//100 yr, yyyymm%100 mo, SUM(exp_dlr) e, SUM(imp_dlr) i
                FROM fact_trade WHERE yyyymm//100 BETWEEN 2007 AND 2025 GROUP BY 1,2),
    t AS (SELECT yr, SUM(e) te, SUM(i) ti FROM ym GROUP BY 1),
    sh AS (SELECT ym.mo, 100.0*ym.e/t.te ep, 100.0*ym.i/t.ti ip FROM ym JOIN t USING(yr))
    SELECT mo AS 월, ROUND(AVG(ep),2) AS 수출비중, ROUND(AVG(ip),2) AS 수입비중
    FROM sh GROUP BY mo ORDER BY mo
''')
print(prof.to_string(index=False))
print("합계 수출:", round(prof['수출비중'].sum(),1), "/ 수입:", round(prof['수입비중'].sum(),1), "(균등=8.33)")

f=plt.figure(figsize=(8,3.6)); ax=f.gca()
ax.plot(range(1,13), prof['수출비중'], marker='o', label='Exports')
ax.plot(range(1,13), prof['수입비중'], marker='s', label='Imports')
ax.axhline(100/12, color='gray', ls='--', lw=.8, label='Even (8.33%)')
ax.set_xticks(range(1,13)); ax.set_xticklabels(MON)
ax.set_ylabel('% of annual total'); ax.set_title('Seasonal Profile of Korea Trade (avg 2007-2025)')
ax.legend(fontsize=7); ax.grid(alpha=.3); f.tight_layout(); plt.show()

## 4. 2월은 왜 낮게 보이는가 — 달력 함정 (평균 월 대비)

In [ ]:
q('''
    WITH ym AS (SELECT yyyymm//100 yr, yyyymm%100 mo, SUM(exp_dlr) e
                FROM fact_trade WHERE yyyymm//100 BETWEEN 2007 AND 2025 GROUP BY 1,2),
    t AS (SELECT yr, AVG(e) avg_month FROM ym GROUP BY 1)
    SELECT ym.mo AS 월, ROUND(AVG(100.0*ym.e/t.avg_month),1) AS 평균월대비_pct
    FROM ym JOIN t USING(yr) GROUP BY ym.mo ORDER BY ym.mo
''')

## 5. 품목마다 계절이 다르다

In [ ]:
# HS2 대분류별 계절 진폭(CV of monthly shares) — 거래액 상위 25개 중
amp = q('''
    WITH ym AS (SELECT SUBSTR(hs10,1,2) hs2, yyyymm//100 yr, yyyymm%100 mo, SUM(exp_dlr) e
                FROM fact_trade WHERE yyyymm//100 BETWEEN 2007 AND 2025 GROUP BY 1,2,3),
    tot AS (SELECT hs2, SUM(e) v FROM ym GROUP BY 1),
    big AS (SELECT hs2 FROM tot ORDER BY v DESC LIMIT 25),
    yt AS (SELECT hs2, yr, SUM(e) te FROM ym GROUP BY 1,2),
    sh AS (SELECT ym.hs2, ym.mo, 100.0*ym.e/yt.te pct FROM ym JOIN yt USING(hs2,yr)
           WHERE ym.hs2 IN (SELECT hs2 FROM big)),
    ms AS (SELECT hs2, mo, AVG(pct) m FROM sh GROUP BY 1,2)
    SELECT hs2 AS HS2, ROUND(STDDEV_SAMP(m)/AVG(m),3) AS 진폭_CV
    FROM ms GROUP BY hs2 ORDER BY 진폭_CV DESC
''')
print("계절적 상위:"); print(amp.head(6).to_string(index=False))
print("\n평탄한 하위:"); print(amp.tail(5).to_string(index=False))

In [ ]:
# 대비 품목(의약품30·전자85·자동차87) 월별 계절 지수(균등=100)
prod = q('''
    WITH ym AS (SELECT SUBSTR(hs10,1,2) hs2, yyyymm//100 yr, yyyymm%100 mo, SUM(exp_dlr) e
                FROM fact_trade WHERE yyyymm//100 BETWEEN 2007 AND 2025
                  AND SUBSTR(hs10,1,2) IN ('30','85','87') GROUP BY 1,2,3),
    yt AS (SELECT hs2, yr, SUM(e) te FROM ym GROUP BY 1,2),
    sh AS (SELECT ym.hs2, ym.mo, 100.0*ym.e/yt.te*12 idx FROM ym JOIN yt USING(hs2,yr))
    SELECT hs2, mo, ROUND(AVG(idx),1) idx FROM sh GROUP BY 1,2 ORDER BY 1,2
''')
piv = prod.pivot(index='mo', columns='hs2', values='idx')
print(piv.to_string())

names={'30':'HS30 Pharma','85':'HS85 Electronics','87':'HS87 Vehicles'}
f=plt.figure(figsize=(8,3.6)); ax=f.gca()
for c in piv.columns:
    ax.plot(range(1,13), piv[c], marker='.', label=names.get(c,'HS'+str(c)))
ax.axhline(100, color='gray', ls='--', lw=.8)
ax.set_xticks(range(1,13)); ax.set_xticklabels(MON)
ax.set_ylabel('Seasonal index (even=100)'); ax.set_title('Seasonality by Product Group (exports)')
ax.legend(fontsize=7); ax.grid(alpha=.3); f.tight_layout(); plt.show()

## 마무리

한국 무역의 계절성은 **전체로는 완만**(2월 저점·연말 고점)하지만, **품목마다 자기 달력**을 가진다 — 의약품·화장품·선박은 계절적, 벌크 화학·철강은 평탄. 모든 셀은 관측된 패턴을 서술할 뿐 인과를 주장하지 않는다. 한계: 완전연도만·2026 제외, 2월 달력 요인 미분해, HS2 기준. 상세는 `chapter02.md` 참조.

In [ ]:
con.close()
print("연결 종료.")